In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

import networkx as nx
import community as community_louvain

### cleaning and encoding labels per method
-> generate anc check the input data

In [2]:

def encode_cluster_labels(df: pd.DataFrame):
    # Drop missing rows (simplest policy)
    print('Rows with Nan values: ', df.isna().any(axis=1).sum())
    df = df.dropna(axis=0, how="any")
    print('Remaining rows: ', df.shape[0])

    spot_ids = df.index.astype(str).tolist()
    methods = df.columns.astype(str).tolist()

    cols = []
    # Encode each column to integer labels
    # returns lists 
    for col in df.columns:
        codes, _ = pd.factorize(df[col], sort=True)
        cols.append(codes.astype(np.int32))
    baseCls = np.stack(cols, axis=1)  # N x M
    return baseCls, spot_ids, methods

# need to make cluster ids globally unique across methods: since build ONE big membership matrix of all methods
def make_global_cluster_ids(baseCls: np.ndarray):
    N, M = baseCls.shape
    nClsOrig = baseCls.max(axis=0) + 1
    offsets = np.concatenate([[0], np.cumsum(nClsOrig)[:-1]]).astype(np.int32)
    bcs = baseCls + offsets
    C_total = int(nClsOrig.sum())
    return bcs, nClsOrig, C_total



### Spatial ESD: building cluster segments 
-> baseClsSegs: number of clusters * number of spots array, binary

### clustering cluster similarity via Jaccard 
-> simxjac matrix

In [3]:
def build_cluster_segments(bcs: np.ndarray, C_total: int):
    N, M = bcs.shape
    rows = bcs.reshape(-1)  # row-major: (spot0 all methods), then spot1...
    cols = np.repeat(np.arange(N, dtype=np.int32), M)  # match row-major
    data = np.ones_like(rows, dtype=np.float32)
    return sp.coo_matrix((data, (rows, cols)), shape=(C_total, N)).tocsr()


def jaccard_similarity_rows(A: sp.csr_matrix) -> np.ndarray:
    """
    Jaccard similarity between rows of sparse binary matrix A (C x N).
    Returns dense (C x C) float array.
    """
    A = A.astype(np.float32)
    inter = (A @ A.T).toarray()  # (C x C)
    sizes = np.asarray(A.sum(axis=1)).reshape(-1)
    union = sizes[:, None] + sizes[None, :] - inter
    with np.errstate(divide="ignore", invalid="ignore"):
        S = np.divide(inter, union, out=np.zeros_like(inter, dtype=np.float32), where=(union > 0))
    return S

### Random walk enhancements
-> compute pts_II matrix: random walk enhancement: turns similarity matrix into smoother, more informative: adding transitivity
- zeroes diagonal
- row-norm to transition matrix P
- accumulating everything
- cosine-normalizing to get newS

In [4]:
def compute_pts_ii(S: np.ndarray, t: int = 20) -> np.ndarray:
    """
    SpatialESD computePTS_II implementation for dense S (C x C).
    """
    S = S.astype(np.float64, copy=True)
    S[S < 0] = 0.0
    np.fill_diagonal(S, 0.0)
    
    row_sum = S.sum(axis=1)
    isolated = row_sum <= 1e-12 # MATLAB uses == 0 before division

    P = np.zeros_like(S)
    P[~isolated] = S[~isolated] / row_sum[~isolated, None]
    P[P < 0] = 0.0  # MATLAB: P(P<0)=0

    tmpP = P.copy()
    inProdP = P @ P.T
    for _ in range(t - 1):
        tmpP = tmpP @ P
        inProdP += tmpP @ tmpP.T

    diag = np.diag(inProdP)
    denom = np.sqrt(np.outer(diag, diag))
    newS = np.divide(inProdP, denom, out=np.zeros_like(inProdP), where=(denom > 0))

    # MATLAB isolation check after normalization
    isolated2 = P.sum(axis=1) < 1e-10
    newS[isolated2, :] = 0.0
    newS[:, isolated2] = 0.0

    return newS.astype(np.float32)


### building similarity spot*spot matrix

In [5]:
def build_method_spot_matrices(bcs: np.ndarray, clsSimRW: np.ndarray) -> list[np.ndarray]:
    """
    For each method m, build A^(m) = clsSimRW[ bcs[:,m], bcs[:,m] ] (N x N).
    Returns list of N x N dense float matrices.
    """
    clsSimRW = clsSimRW.copy()
    np.fill_diagonal(clsSimRW, 1.0)

    N, M = bcs.shape
    mats = []
    for m in range(M):
        c = bcs[:, m]
        A = clsSimRW[np.ix_(c, c)].astype(np.float32, copy=False)
        mats.append(A)
    return mats


def frobenius_norm(X: np.ndarray) -> float:
    return float(np.sqrt(np.sum(X * X)))

def ensdd_weighted_consensus(
    mats: list[np.ndarray],
    lambda_: float | None = None,
    prob_quantile: float = 0.5,
    niter: int = 100,
    epsilon: float = 1e-5
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Returns:
      H: consensus matrix (N x N)
      w: weights (M,)
      info: dict with loss history etc.
    """
    M = len(mats)
    if M == 0:
        raise ValueError("No method matrices provided.")
    N = mats[0].shape[0]
    for A in mats:
        if A.shape != (N, N):
            raise ValueError("All matrices must have same shape (N,N).")

    w = np.full(M, 1.0 / M, dtype=np.float64)
    H = sum(w[m] * mats[m] for m in range(M)).astype(np.float64)

    losses = []
    lambdas = []
    diffs = []

    prev_loss = None
    for k in range(1, niter + 1):
        d = np.array([frobenius_norm(mats[m].astype(np.float64) - H) for m in range(M)], dtype=np.float64)

        if k == 1 and lambda_ is None:
            lambda_ = float(np.quantile(d, prob_quantile))
            if lambda_ <= 0:
                lambda_ = 1e-8

        z = -d / lambda_
        z -= z.max()
        w = np.exp(z)
        w /= w.sum()

        # update H
        H = sum(w[m] * mats[m] for m in range(M)).astype(np.float64)

        # recompute distances for updated H  
        d_new = np.array([frobenius_norm(mats[m].astype(np.float64) - H) for m in range(M)], dtype=np.float64)

        loss_main = float(np.sum(w * d_new))
        loss_entropy = float(np.sum(w * np.log(np.clip(w, 1e-15, 1.0))))
        loss_all = loss_main + lambda_ * loss_entropy
        losses.append(loss_all)
        lambdas.append(lambda_)
        if prev_loss is None:
            diffs.append(np.nan)
        else:
            diffs.append(abs(loss_all - prev_loss))

        if prev_loss is not None and abs(loss_all - prev_loss) < epsilon:
            break
        prev_loss = loss_all

    #info = {"loss": np.array(losses), "lambda": np.array(lambdas), "diff": np.array(diffs)}
    return H.astype(np.float32), w.astype(np.float32)

def dense_similarity_to_csr_knn(
    H: np.ndarray,
    k: int = 30,
    keep_diagonal: bool = False,
    make_symmetric: bool = True,
    dtype=np.float32
) -> sp.csr_matrix:
    """
    Build a kNN graph (per row keep top-k similarities).
    Produces a sparse CSR suitable for Leiden.
    """
    H = np.asarray(H)
    N = H.shape[0]

    if make_symmetric:
        H = 0.5 * (H + H.T)

    if not keep_diagonal:
        np.fill_diagonal(H, -np.inf)

    rows = []
    cols = []
    data = []

    for i in range(N):
        row = H[i]
        # indices of top-k (excluding -inf)
        idx = np.argpartition(row, -k)[-k:]
        # filter out invalid
        idx = idx[np.isfinite(row[idx])]
        rows.extend([i] * len(idx))
        cols.extend(idx.tolist())
        data.extend(row[idx].astype(dtype, copy=False).tolist())

    A = sp.coo_matrix((np.array(data, dtype=dtype),
                       (np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32))),
                      shape=(N, N)).tocsr()

    # symmetrize by max (common for neighbor graphs)
    if make_symmetric:
        A = A.maximum(A.T)

    # set diagonal to 0
    A.setdiag(0.0)
    A.eliminate_zeros()
    return A

### edge based approach functions for big datasets
- new implementation as possibility  

In [6]:
import numpy as np
import scipy.sparse as sp

def edge_based_weighted_consensus(
    labels_by_method,        # (N, M) int array, already encoded per method: needs to be bcs (global!)
    clsSimRW,                # (C, C) enhanced cluster similarity (global cluster ids)
    A_mask,                  # sparse adjacency mask (N, N) CSR (where edges exist)
    lam=None,
    prob_quantile=0.5,
    niter=100,
    eps=1e-5,
    dtype=np.float32,
):
    """
    Returns:
      H_masked: CSR sparse (N,N) with weights only on edges of A_mask
      w: (M,) weights
    """
    A_mask = A_mask.tocsr()
    A_mask.eliminate_zeros()

    # use COO to get edge list once
    Acoo = A_mask.tocoo()
    i, j = Acoo.row, Acoo.col

    # keep only upper triangle to avoid duplicates
    keep = i < j
    i, j = i[keep], j[keep]
    E = len(i)

    N, M = labels_by_method.shape
    
    clsSimRW = clsSimRW.copy()
    np.fill_diagonal(clsSimRW, 1.0)

    # build per-method edge weights a_m[e]
    A_edges = np.empty((M, E), dtype=dtype)
    for m in range(M):
        cm_i = labels_by_method[i, m]
        cm_j = labels_by_method[j, m]
        A_edges[m] = clsSimRW[cm_i, cm_j].astype(dtype)

    # init uniform weights
    w = np.ones(M, dtype=dtype) / M
    H = (w[:, None] * A_edges).sum(axis=0)

    # choose lambda like EnSDD
    dists = np.sqrt(((A_edges - H[None, :]) ** 2).sum(axis=1))
    if lam is None:
        lam = np.quantile(dists, prob_quantile).astype(dtype)
        lam = max(lam, dtype(1e-8))

    last = None
    for it in range(niter):
        dists = np.sqrt(((A_edges - H[None, :]) ** 2).sum(axis=1))

        logits = -dists / lam
        logits -= logits.max()
        w_new = np.exp(logits).astype(dtype)
        w_new /= w_new.sum()

        H_new = (w_new[:, None] * A_edges).sum(axis=0)

        # recompute distances for the *new* H (matches EnSDD)
        dists_new = np.sqrt(((A_edges - H_new[None, :]) ** 2).sum(axis=1))

        loss = (w_new * dists_new).sum() + lam * (w_new * np.log(w_new + 1e-12)).sum()

        if last is not None and abs(loss - last) < eps:
            w, H = w_new, H_new
            break

        w, H, last = w_new, H_new, loss
        
    H_masked = sp.coo_matrix((H, (i, j)), shape=A_mask.shape).tocsr()
    H_masked = H_masked.maximum(H_masked.T)
    H_masked.setdiag(0.0)
    H_masked.eliminate_zeros()


    return H_masked, w


### ENSDD style weight optimization + regularization + consensus clustering: for dense!

### clustering meethods finalizing:
Leiden: 2 option:
- give in resolution parameter
- give in exactly number k that should be output as clusters
- give in compute best resolution parameter to get optimal via NNL and BIC

In [7]:
import numpy as np
import scipy.sparse as sp

def csr_to_igraph(A: sp.csr_matrix):
    import igraph as ig
    A = A.tocsr()
    A.eliminate_zeros()

    # use only upper triangle edges
    A_triu = sp.triu(A, k=1).tocoo()
    edges = list(zip(A_triu.row.tolist(), A_triu.col.tolist()))
    weights = A_triu.data.astype(float).tolist()

    g = ig.Graph(n=A.shape[0], edges=edges, directed=False)
    g.es["weight"] = weights
    return g


def leiden_from_csr(A: sp.csr_matrix, resolution: float = 1.0, seed: int = 0,
                    partition_type: str = "rb"):
    """
    partition_type:
      - "rb": RBConfigurationVertexPartition (common, like 'resolution' modularity)
      - "cpm": CPMVertexPartition (often more stable for controlling cluster sizes)
    """
    import leidenalg as la

    # enforce symmetry + clean diagonal
    A = A.maximum(A.T).tocsr()
    A.setdiag(0.0)
    A.eliminate_zeros()

    g = csr_to_igraph(A)

    if partition_type == "rb":
        part = la.find_partition(
            g, la.RBConfigurationVertexPartition,
            weights=g.es["weight"],
            resolution_parameter=float(resolution),
            seed=int(seed),
        )
    elif partition_type == "cpm":
        part = la.find_partition(
            g, la.CPMVertexPartition,
            weights=g.es["weight"],
            resolution_parameter=float(resolution),
            seed=int(seed),
        )
    else:
        raise ValueError("partition_type must be 'rb' or 'cpm'")

    labels = np.asarray(part.membership, dtype=np.int32)
    return labels


In [8]:
# for specified k
def tune_resolution_for_k(A: sp.csr_matrix, K: int,
                          r_min: float = 0.01, r_max: float = 5.0,
                          max_iter: int = 30, tolerance: int = 0,
                          seed: int = 0, partition_type: str = "rb"):
    """
    Returns: labels, resolution_found, history(list of dicts)
    """
    history = []
    best = None  # (abs_diff, resolution, labels)

    lo, hi = float(r_min), float(r_max)
    for it in range(max_iter):
        mid = (lo + hi) / 2.0
        labels = leiden_from_csr(A, resolution=mid, seed=seed, partition_type=partition_type)
        k_mid = int(np.unique(labels).size)
        diff = abs(k_mid - K)

        history.append({"iter": it, "resolution": mid, "k": k_mid, "diff": diff})

        if best is None or diff < best[0]:
            best = (diff, mid, labels)

        # stop if within tolerance
        if k_mid >= K - tolerance and k_mid <= K + tolerance:
            return labels, mid, history

        # adjust bounds
        if k_mid > K + tolerance:
            hi = mid
        else:
            lo = mid

    # fallback: closest we saw
    return best[2], best[1], history



In [9]:
# for best estimation
def sbm_nll_bic_on_edges(A: sp.csr_matrix, labels: np.ndarray, eps: float = 1e-12):
    """
    Compute NLL and BIC using only observed edges (upper triangle).
    A: sparse weighted adjacency (CSR)
    labels: 0..K-1
    """
    A = A.maximum(A.T).tocsr()
    A.setdiag(0.0)
    A.eliminate_zeros()

    # edges
    E = sp.triu(A, k=1).tocoo()
    i, j, w = E.row, E.col, E.data.astype(np.float64)
    n_obs = len(w)
    if n_obs == 0:
        return np.inf, np.inf, None

    labs = labels.astype(np.int32)
    K = int(np.unique(labs).size)

    # map cluster ids to 0..K-1 (in case)
    uniq = np.unique(labs)
    remap = {c: idx for idx, c in enumerate(uniq)}
    gi = np.array([remap[x] for x in labs[i]], dtype=np.int32)
    gj = np.array([remap[x] for x in labs[j]], dtype=np.int32)

    # estimate theta_ab as mean edge weight in block (a,b)
    # accumulate sums/counts for symmetric blocks
    sums = np.zeros((K, K), dtype=np.float64)
    cnts = np.zeros((K, K), dtype=np.float64)

    for a, b, ww in zip(gi, gj, w):
        if a <= b:
            sums[a, b] += ww
            cnts[a, b] += 1.0
        else:
            sums[b, a] += ww
            cnts[b, a] += 1.0

    theta = sums / np.maximum(cnts, 1.0)
    theta = np.clip(theta, eps, 1.0)

    # NLL over edges: -sum log(theta_block)
    nll = 0.0
    for a, b in zip(gi, gj):
        if a <= b:
            nll -= np.log(theta[a, b])
        else:
            nll -= np.log(theta[b, a])

    # BIC penalty
    p = K * (K + 1) / 2.0
    bic = 2.0 * nll + p * np.log(max(n_obs, 2))

    return float(nll), float(bic), theta


def select_resolution_by_bic(A: sp.csr_matrix,
                             resolutions=None,
                             seed: int = 0,
                             partition_type: str = "rb"):
    """
    Scan resolutions, compute Leiden labels, NLL, BIC, pick best (min BIC).
    Returns: best_labels, best_res, table(list of dicts)
    """
    if resolutions is None:
        # reasonable default scan
        resolutions = np.linspace(0.05, 3.0, 30)

    table = []
    best = None  # (bic, res, labels)

    for res in resolutions:
        labels = leiden_from_csr(A, resolution=float(res), seed=seed, partition_type=partition_type)
        nll, bic, _ = sbm_nll_bic_on_edges(A, labels)
        k = int(np.unique(labels).size)

        row = {"resolution": float(res), "k": k, "nll": nll, "bic": bic}
        table.append(row)

        if best is None or bic < best[0]:
            best = (bic, float(res), labels)

    return best[2], best[1], table


In [10]:
def leiden_final(A: sp.csr_matrix,
                 resolution: float | None = None,
                 K: int | None = None,
                 auto_bic: bool = False,
                 seed: int = 0,
                 partition_type: str = "rb"):
    """
    Choose exactly one mode:
      - resolution provided -> mode 1
      - K provided -> mode 2
      - auto_bic=True -> mode 3
    Returns: labels, info(dict)
    """
    if sum([resolution is not None, K is not None, auto_bic]) != 1:
        raise ValueError("Choose exactly one of: resolution, K, auto_bic=True")

    if resolution is not None:
        labels = leiden_from_csr(A, resolution=resolution, seed=seed, partition_type=partition_type)
        return labels, {"mode": "resolution", "resolution": float(resolution),
                        "k": int(np.unique(labels).size)}

    if K is not None:
        labels, res_found, hist = tune_resolution_for_k(
            A, K=K, seed=seed, partition_type=partition_type
        )
        return labels, {"mode": "target_k", "resolution": float(res_found),
                        "k": int(np.unique(labels).size), "history": hist}

    # auto bic
    labels, res_best, table = select_resolution_by_bic(
        A, seed=seed, partition_type=partition_type
    )
    return labels, {"mode": "auto_bic", "resolution": float(res_best),
                    "k": int(np.unique(labels).size), "scan": table}


In [28]:
# now all together: read in to csv files

tool_domains_dir = "/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/examples_csvs"
# for all csvs in dir: read in each csv (only one column per csv) to dataframe to use
import os
csv_files = [f for f in os.listdir(tool_domains_dir) if f.endswith('.csv')]
tools_df = []
# see if len are identical
lengths = []
for f in csv_files:
    df = pd.read_csv(os.path.join(tool_domains_dir, f), header=None, index_col=None)
    print(f)
    print(df)
    print(df.shape)
    lengths.append(df.shape[0])
    method_name = f.split('.csv')[0]
    df.columns = [method_name]
    # set index first to 1: so index starting from 1
    df.index = range(1, df.shape[0] + 1)
    tools_df.append(df)
    
combined_df = pd.concat(tools_df, axis=1)
if len(set(lengths)) != 1:
    raise ValueError("CSV files have different number of rows!")

combined_df


groundtruth.csv
      0
0     1
1     2
2     3
3     4
4     5
..   ..
699   2
700   2
701  10
702   2
703   2

[704 rows x 1 columns]
(704, 1)
stlearn.csv
      0
0     1
1     2
2     3
3     4
4     5
..   ..
699   4
700   4
701  10
702   2
703   1

[704 rows x 1 columns]
(704, 1)
graphst.csv
      0
0     1
1     2
2     3
3     4
4     5
..   ..
699   3
700   3
701  10
702   2
703   1

[704 rows x 1 columns]
(704, 1)


,groundtruth,stlearn,graphst
1,1,1,1
2,2,2,2
3,3,3,3
4,4,4,4
5,5,5,5
...,...,...,...
700,2,4,3
701,2,4,3
702,10,10,10
703,2,2,2


In [ ]:
# use all predefined methods
print('Starting Ensemble calculation...')
baseCls, spot_ids, methods = encode_cluster_labels(combined_df)
print('methods: ', methods)
print('finished encoding cluster labels\n')
bcs, nClsOrig, C_total = make_global_cluster_ids(baseCls)
print('finished making global cluster ids\n')
cluster_segments = build_cluster_segments(bcs, C_total)
print('finished building cluster segments\n')
clsSim = jaccard_similarity_rows(cluster_segments)
print('finished computing cluster similarities: Jaccard\n')
clsSimRW = compute_pts_ii(clsSim, t=20)
print('finished computing cluster similarities: Random walk\n')
method_spot_mats = build_method_spot_matrices(bcs, clsSimRW)
print("Number of method spot matrices: ", len(method_spot_mats))
for idx, mat in enumerate(method_spot_mats):
    print(f"Method {methods[idx]} spot matrix shape: ", mat.shape)
H, w = ensdd_weighted_consensus(method_spot_mats, niter=100, epsilon=1e-5)
print("Consensus matrix H shape: ", H.shape)
print('finished computing consensus matrix\n')
print("Method weights: ", w)
H_csr = dense_similarity_to_csr_knn(H, k=5, keep_diagonal=False, make_symmetric=True, dtype=np.float32)
print('finished building consensus kNN graph\n')
labels, info = leiden_final(H_csr, K=12, seed=0, partition_type="rb")
print('finished Leiden clustering with \n')
print("Leiden info: ", info)

Rows with Nan values:  0
Remaining rows:  704
Base cluster shape:  (704, 3)
methods:  ['groundtruth', 'stlearn', 'graphst']
Global cluster shape:  (704, 3)
Cluster segments shape:  (36, 704)
Cluster segments:  <Compressed Sparse Row sparse matrix of dtype 'float32'
	with 2112 stored elements and shape (36, 704)>
  Coords	Values
  (0, 0)	1.0
  (0, 14)	1.0
  (0, 18)	1.0
  (0, 19)	1.0
  (0, 20)	1.0
  (0, 21)	1.0
  (0, 28)	1.0
  (0, 33)	1.0
  (0, 41)	1.0
  (0, 56)	1.0
  (0, 63)	1.0
  (0, 64)	1.0
  (0, 66)	1.0
  (0, 91)	1.0
  (0, 96)	1.0
  (0, 112)	1.0
  (0, 121)	1.0
  (0, 138)	1.0
  (0, 139)	1.0
  (0, 140)	1.0
  (0, 149)	1.0
  (0, 168)	1.0
  (0, 171)	1.0
  (0, 174)	1.0
  (0, 181)	1.0
  :	:
  (35, 427)	1.0
  (35, 436)	1.0
  (35, 440)	1.0
  (35, 446)	1.0
  (35, 447)	1.0
  (35, 450)	1.0
  (35, 472)	1.0
  (35, 480)	1.0
  (35, 509)	1.0
  (35, 520)	1.0
  (35, 529)	1.0
  (35, 531)	1.0
  (35, 548)	1.0
  (35, 556)	1.0
  (35, 571)	1.0
  (35, 585)	1.0
  (35, 589)	1.0
  (35, 597)	1.0
  (35, 609)	1.0
 

In [36]:
# leiden clustering to format csv as the others
import pandas as pd
leiden_df = pd.DataFrame(labels, index=spot_ids, columns=['leiden_clustering'])
leiden_df

,leiden_clustering
1,2
2,4
3,0
4,11
5,7
...,...
700,4
701,4
702,9
703,4


In [37]:
# to csv
path_ensemble = os.path.join(tool_domains_dir, 'ensemble.csv')
leiden_df.to_csv(path_ensemble, header=False, index=False)

In [34]:
C = clsSimRW.shape[0]
off = clsSimRW[~np.eye(C, dtype=bool)]
print(off.min(), off.mean(), np.quantile(off, [0.5, 0.9, 0.99]))


0.0 0.07384643 [0.00172141 0.0991731  0.95083886]
